In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import pandas as pd
train_dir = '/content/asl_alphabet_test.zip'
test_dir = '/content/asl_alphabet_test.zip'


import cv2
import os
import numpy as np
def get_data(train_path):
 X = []
 Y = []
 for dir in os.listdir(train_path):
   dir_path = train_path + "/" + dir
   print(f"{dir} processed")
   for image_name in  os.listdir(dir_path):
    image_path = dir_path + "/" + image_name
    image = cv2.imread(image_path)
    image_color = cv2.cvtColor(image,cv2.COLOR_BGR2RGB)
    image_resized = cv2.resize(image_color,(32,32))
    X.append(image_resized)
    Y.append(dir)
 return np.array(X),np.array(Y)


In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import keras

train_datagen=ImageDataGenerator(
    rescale=1./255,
    rotation_range=10,
    zoom_range=0.1,
    width_shift_range=0.1,
    height_shift_range=0.1
)
train_generator=train_datagen.flow(
    train_images,
    keras.utils.to_categorical(train_labels,26),
    batch_size=32
)

test_datagen=ImageDataGenerator(
    rescale=1./255,
)
test_generator=test_datagen.flow(
    test_images,
    keras.utils.to_categorical(test_labels,26),
    batch_size=32
)

In [ ]:
import keras

model=keras.Sequential([
    keras.layers.Input(shape=(32,32,3)),
    keras.layers.Conv2D(32,(4,4),activation='relu'),
    keras.layers.MaxPooling2D(),
    keras.layers.Conv2D(64,(4,4),activation='relu'),
    keras.layers.MaxPooling2D(),
    keras.layers.Flatten(),
    keras.layers.Dense(256,activation='relu'),
    keras.layers.Dense(26,activation='softmax')
])
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
model.summary()


In [ ]:
from keras.callbacks import Callback
class call(Callback):
    def on_epoch_end(self,epoch,logs={}):
        if logs.get('val_accuracy')>0.99:
            print('\nReashed +99% val_accuracy')
            self.model.stop_training=True

In [ ]:
history=model.fit(
    train_generator,
    epochs=10,
    validation_data=test_generator,
    callbacks=[call()]
)

In [ ]:
X.shape

In [ ]:
print(len(X), len(y))

In [ ]:
model.evaluate(test_generator)

In [ ]:
plt.figure(figsize=(12, 5))

# Plot loss
plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Training and Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

# Plot accuracy
plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Training and Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
from keras.applications import ResNet50
base_model = ResNet50(weights=None, include_top=False, input_shape=(32, 32, 3))
from keras.models import Model, Sequential
from keras.layers import Dense, Flatten, Dropout, LSTM, Input
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from keras.utils import to_categorical
from keras.callbacks import Callback
x = Flatten()(base_model.output)
x = Dense(256, activation='relu')(x)
x = Dropout(0.3)(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.2)(x)
output = Dense(26, activation='softmax')(x)
resnet_model = Model(inputs=base_model.input, outputs=output)
resnet_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

In [ ]:
lstm_model = Sequential([
    Input(shape=(28, 28)),
    LSTM(128, return_sequences=True),
    Dropout(0.3),
    LSTM(64, return_sequences=False),
    Dropout(0.2),
    Dense(256, activation='relu'),
    Dense(26, activation='softmax')
])
lstm_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])


In [ ]:
# Train Models
epochs = 10
history_resnet = resnet_model.fit(train_generator, epochs=epochs, validation_data=test_generator, callbacks=[call()])
history_lstm = lstm_model.fit(train_images.reshape(-1, 32, 32), train_labels, epochs=epochs, validation_data=(test_images.reshape(-1, 32, 32), test_labels), callbacks=[call()])


In [ ]:
# Evaluate Models
eval_resnet = resnet_model.evaluate(test_generator)
eval_lstm = lstm_model.evaluate(test_images.reshape(-1, 28, 28), test_labels)

In [ ]:
# Plot Accuracy & Loss
def plot_metrics(history, model_name):
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].plot(history.history['accuracy'], label='Train Accuracy')
    ax[0].plot(history.history['val_accuracy'], label='Validation Accuracy')
    ax[0].set_title(f'{model_name} - Accuracy')
    ax[0].set_xlabel('Epochs')
    ax[0].set_ylabel('Accuracy')
    ax[0].legend()

    ax[1].plot(history.history['loss'], label='Train Loss')
    ax[1].plot(history.history['val_loss'], label='Validation Loss')
    ax[1].set_title(f'{model_name} - Loss')
    ax[1].set_xlabel('Epochs')
    ax[1].set_ylabel('Loss')
    ax[1].legend()

    plt.show()

plot_metrics(history_resnet, "ResNet-50 Model")
plot_metrics(history_lstm, "LSTM Model")
